In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

print("=== STARTING WEEK 10: ADVANCED SALES & CUSTOMER ANALYTICS ===")

np.random.seed(42)
n_customers = 1000

customer_ids = [f"CUST-{1000 + i}" for i in range(n_customers)]
recency_days = np.random.exponential(scale=45, size=n_customers).astype(int)
frequency_orders = np.random.poisson(lam=5, size=n_customers) + 1
monetary_value = frequency_orders * np.random.normal(loc=150, scale=40, size=n_customers)
monetary_value = np.maximum(monetary_value, 20.0)

df_rfm = pd.DataFrame({
    'CustomerID': customer_ids,
    'Recency': recency_days,
    'Frequency': frequency_orders,
    'Monetary': monetary_value
})

df_rfm['R_Score'] = pd.qcut(df_rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])
df_rfm['F_Score'] = pd.qcut(df_rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
df_rfm['M_Score'] = pd.qcut(df_rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5])

def segment_customer(df):
    r = int(df['R_Score'])
    f = int(df['F_Score'])
    m = int(df['M_Score'])
    
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 3 and f <= 2:
        return 'Recent / New'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    else:
        return 'Dormant'

df_rfm['Segment'] = df_rfm.apply(segment_customer, axis=1)

segment_summary = df_rfm.groupby('Segment').agg(
    Customer_Count=('CustomerID', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Total_Revenue=('Monetary', 'sum')
).reset_index()

print(segment_summary.to_string(index=False))
segment_summary.to_csv('../analysis/rfm_segmentation_summary.csv', index=False)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.barplot(data=segment_summary, x='Segment', y='Total_Revenue', ax=axes[0, 0], palette='viridis')
axes[0, 0].set_title('Total Revenue Contribution by Customer Segment')
sns.barplot(data=segment_summary, x='Segment', y='Customer_Count', ax=axes[0, 1], palette='mako')
axes[0, 1].set_title('Customer Distribution Across Segments')
sns.scatterplot(data=df_rfm, x='Recency', y='Monetary', hue='Segment', ax=axes[1, 0], alpha=0.7, palette='Set2')
axes[1, 0].set_title('Recency vs Monetary Value Dispersion')
sns.histplot(df_rfm['Frequency'], bins=15, kde=True, ax=axes[1, 1], color='teal')
axes[1, 1].set_title('Customer Order Frequency Distribution')

plt.tight_layout()
plt.show()
print("=== WEEK 10 ANALYSIS COMPLETE ===")